In [ ]:
import pandas as pd

# 22k url, missing scheme, one column, no header
cybercrimeTracker_path = '../../datasets/malicious/cybercrimeTracker'
# folder path, contains three folders, each contains files with urls, ips...
# go through all the csv files, read them with comment='#' and no header, filter urls
maltrial_path = '../../datasets/malicious/maltrail'
# sophoslabs folder path, contains many csv files, with domains, ips...
# go through all the csv files and extract urls, also clean them (ex: https[//:]... => https//:...)
sophoslabs_path = '../../datasets/malicious/sophoslabs'
# use threatfox urls as is and extract the urls from the full file

In [43]:
cybercrimeTracker_df = pd.read_csv(f'{cybercrimeTracker_path}/CYBERCRiME-02-11-26.txt', header=None, names=['url'])

cybercrimeTracker_df

,url
0,5.182.86.32/auth/login
1,79.137.194.188/auth/login
2,achillharpfestival.ie/wp-content/plugins/dbzyt...
3,achillharpfestival.ie/wp-content/plugins/dbzyt...
4,unwaveringmedia.com/wp-content/plugins/vsqjspm...
...,...
22695,strelokfancy.com/viaweb/control.php
22696,85.95.247.26/~estacion/Panel/Web-Panel/priv8/
22697,194.146.227.48:8080/pony/admin.php
22698,176.31.255.41:81/pony/admin.php


In [ ]:
# resolve the scheme of the cybercrimeTracker urls
import requests
import ftplib
from urllib.parse import urlparse
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

def resolve_scheme(partial_url, timeout=15):
	"""
	Detect correct scheme by testing protocols.
	Returns full URL with scheme, or None if all fail.
	"""
	# Extract domain properly
	parsed = urlparse(f"http://{partial_url}")
	domain = parsed.netloc or partial_url.split('/')[0]
	
	# Try HTTP/HTTPS
	for scheme in ['https', 'http']:
		try:
			response = requests.head(
				f"{scheme}://{partial_url}",
				timeout=timeout,
				allow_redirects=True,
				verify=False,
				headers={"User-Agent": "Mozilla/5.0"}
			)
			# Any valid HTTP response means scheme works
			if response.status_code < 600:
				return f"{scheme}://{partial_url}"
		except:
				continue
	
	# Try FTP
	try:
		ftp = ftplib.FTP(timeout=timeout)
		ftp.connect(domain, 21, timeout=timeout)
		ftp.quit()
		return f"ftp://{partial_url}"
	except:
		pass
	
	# All checks failed - URL is dead/unreachable
	return None


In [17]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

results = [None] * len(cybercrimeTracker_df)
with ThreadPoolExecutor(max_workers=100) as executor:
	futures = {executor.submit(resolve_scheme, cybercrimeTracker_df.loc[i, 'url']): i for i in range(len(cybercrimeTracker_df))}
	for future in tqdm(as_completed(futures), total=len(futures), desc="Resolving chunk"):
		idx = futures[future]
		results[idx] = future.result()

Resolving chunk: 100%|██████████| 22700/22700 [1:25:52<00:00,  4.41it/s]  


In [22]:
cybercrimeTracker_df['resolved_url'] = results

cybercrimeTracker_df

,url,resolved_url
0,5.182.86.32/auth/login,https://5.182.86.32/auth/login
1,79.137.194.188/auth/login,https://79.137.194.188/auth/login
2,achillharpfestival.ie/wp-content/plugins/dbzyt...,https://achillharpfestival.ie/wp-content/plugi...
3,achillharpfestival.ie/wp-content/plugins/dbzyt...,https://achillharpfestival.ie/wp-content/plugi...
4,unwaveringmedia.com/wp-content/plugins/vsqjspm...,http://unwaveringmedia.com/wp-content/plugins/...
...,...,...
22695,strelokfancy.com/viaweb/control.php,NaN
22696,85.95.247.26/~estacion/Panel/Web-Panel/priv8/,NaN
22697,194.146.227.48:8080/pony/admin.php,NaN
22698,176.31.255.41:81/pony/admin.php,NaN


In [42]:
attempt2_df = cybercrimeTracker_df[cybercrimeTracker_df['resolved_url'].isna()][['url']].reset_index(drop=True)

attempt2_df

,url
0,103.147.185.68/o2/login.php
1,103.147.185.68/l3/login.php
2,103.147.185.68/i1/login.php
3,103.147.185.68/k4/login.php
4,103.147.185.68/j/p28ua/login.php
...,...
19211,strelokfancy.com/viaweb/control.php
19212,85.95.247.26/~estacion/Panel/Web-Panel/priv8/
19213,194.146.227.48:8080/pony/admin.php
19214,176.31.255.41:81/pony/admin.php


In [ ]:
# use VPN in the second attempt to resolved banned domains in your region
attempt2_results = [None] * len(attempt2_df)
with ThreadPoolExecutor(max_workers=100) as executor:
	futures = {executor.submit(resolve_scheme, attempt2_df.loc[i, 'url']): i for i in range(len(attempt2_df))}
	for future in tqdm(as_completed(futures), total=len(futures), desc="Resolving chunk"):
		idx = futures[future]
		attempt2_results[idx] = future.result()

Resolving chunk: 100%|██████████| 19216/19216 [31:08<00:00, 10.28it/s]  


In [43]:
attempt2_df['resolved_url'] = attempt2_results

attempt2_df

,url,resolved_url
0,103.147.185.68/o2/login.php,NaN
1,103.147.185.68/l3/login.php,NaN
2,103.147.185.68/i1/login.php,NaN
3,103.147.185.68/k4/login.php,NaN
4,103.147.185.68/j/p28ua/login.php,NaN
...,...,...
19211,strelokfancy.com/viaweb/control.php,NaN
19212,85.95.247.26/~estacion/Panel/Web-Panel/priv8/,NaN
19213,194.146.227.48:8080/pony/admin.php,NaN
19214,176.31.255.41:81/pony/admin.php,NaN


In [ ]:
pd.concat([
  cybercrimeTracker_df.loc[cybercrimeTracker_df['resolved_url'].notna(), ['resolved_url']],
  attempt2_df.loc[attempt2_df['resolved_url'].notna(), ['resolved_url']]
  ], 
	ignore_index=True
)\
  .rename(columns={'resolved_url': 'url'})\
  .to_csv(f'{cybercrimeTracker_path}/resolved.csv')


In [20]:
# maltrial
import os

chunks = []

for folder in os.listdir(maltrial_path):
	if all([extension not in folder for extension in ['.csv', '.txt', 'other file extensions']]):
		for file in os.listdir(f'{maltrial_path}/{folder}'):
			if file.endswith('.txt'):
				try:
					chunks.append(pd.read_csv(f'{maltrial_path}/{folder}/{file}', comment='#', header=None, names=['url'], sep='\t'))
				except:
					print(f'{maltrial_path}/{folder}/{file}')

# should be cleaned
maltrial_df = pd.concat(chunks, ignore_index=True)
maltrial_df

,url
0,aps.kemoge.net
1,taosha.cc
2,123.194.235.37:49320
3,1.34.220.200:52672
4,186.188.229.46:44977
...,...
1324350,dauled.com
1324351,erbolsan.com
1324352,kasym500.com
1324353,kasymdev.com


In [22]:
maltrial_df.to_csv(f'{maltrial_path}/all.csv', columns=["url"])

In [49]:
# sophoslabs
chunks = []

for file in os.listdir(sophoslabs_path):
  if file.endswith('.csv'):
    try:
      chunks.append(pd.read_csv(f'{sophoslabs_path}/{file}', header=None, skiprows=1))
    except Exception as e:
      print(e)
      print(f'{sophoslabs_path}/{file}')

sophoslabs_df = pd.concat(chunks, ignore_index=True)
sophoslabs_df


,0,1,2
0,description,NaN,From the story https://news.sophos.com/en-us/2...
1,domain,abdurantom.com,FakeAlert scam domain
2,domain,airwiseq.cf,FakeAlert host
3,domain,appconnect.bekapro.xyz,FakeAlert host
4,domain,apps-notification.com,FakeAlert host
...,...,...,...
17959,sha256,0a671d9d7ca62274e5e210813d02d860846baf59188e2a...,RDP backdoor
17960,sha256,2c44444d207a78da7477ae1af195d4265134e895bebb47...,Mount Locker ransomware DLL
17961,scheduled task,regsvr32.exe /i C:\Users\<username>\AppData\wi...,Name: updater
17962,scheduled task,regsvr32.exe /i C:\Program Files\Google\Drive\...,Name: updater


In [50]:
sophoslabs_df.columns = ['indicator_type', 'data', 'note']
sophoslabs_df.to_csv(f'{sophoslabs_path}/all.csv')

In [ ]:
# this script will fetch pulses that you are subscribed to (you need to create an alienvault account and get your own api key), iterate through the pulses and extracts indicator if it's a url
from OTXv2 import OTXv2
import time

otx = OTXv2("YOUR_API_KEY")

urls = set()

# Phase 1: Get subscribed pulses
try:
	subscribed = otx.getall()
	for pulse in subscribed:
		for ind in pulse.get('indicators', []):
			if ind.get('type') in ['URL', 'URI']:
				urls.add(ind['indicator'])
	print(f"getting subscribed pulses complete: {len(urls):,} URLs\n")
except Exception as e:
	print(f"error getting subscribed pulses: {e}\n")
 
with open('otx_urls_subscribed.txt', 'a') as file:
  for url in urls:
    file.write(f'{url}\n')


getting subscribed pulses complete: 38,809 URLs



In [ ]:
# this script will look for pulses matching a set of keywords, iterate through the pulses and extracts indicator if it's a url
from OTXv2 import OTXv2
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

otx = OTXv2("YOUR_API_KEY")

def get_pulse_details(id):
  try:
    return otx.get_pulse_details(id)
  except:
    return None

urls = set()

malware_families = [
    'remcos', 'malspam', 'ransomware', 'banker', 'stealer', "botnet", "stealer", "banking trojan",
    'malicious', 'url', 'malware'
]

for family in malware_families:
	try:
		print(f"Searching {family}...")
		results = otx.search_pulses(family, max_results=500)
		
		# Handle different response formats
		if isinstance(results, dict):
			pulse_list = results.get('results', [])
		elif isinstance(results, list):
			pulse_list = results
		else:
			print(f"  Unexpected format: {type(results)}")
			continue

		family_urls = 0
		results = [None] * len(pulse_list)
		with ThreadPoolExecutor(max_workers=100) as executor:
			futures = {executor.submit(get_pulse_details, pulse.get('id') if isinstance(pulse, dict) else pulse): pulse for pulse in pulse_list}
			for future in tqdm(as_completed(futures), total=len(futures), desc="Resolving pulses"):
				for ind in future.result().get('indicators', []):
					if ind.get('type') in ['URL', 'URI']:
						urls.add(ind['indicator'])
						family_urls += 1

		print(f'{family_urls} for {family}')
		time.sleep(3)  # Rate limit between families
			
	except Exception as e:
		print(f"  Error with {family}: {e}")

print(f"\n{'='*60}")
print(f"✓ COLLECTION COMPLETE")
print(f"{'='*60}")
print(f"Total URLs collected: {len(urls):,}")

with open('otx_urls_searched.txt', 'a') as file:
  for url in urls:
    file.write(f'{url}\n')

Searching remcos...


Resolving pulses:  98%|█████████▊| 489/500 [03:39<00:04,  2.22it/s] 


  Error with remcos: 'NoneType' object has no attribute 'get'
Searching malspam...


Resolving pulses: 100%|██████████| 500/500 [00:06<00:00, 78.92it/s] 


69 for malspam
Searching ransomware...


...

373 for malicious
Searching url...


Resolving pulses: 100%|██████████| 500/500 [00:11<00:00, 43.38it/s] 


2623 for url
Searching malware...


Resolving pulses: 100%|██████████| 500/500 [00:16<00:00, 29.66it/s] 


1084 for malware

✓ COLLECTION COMPLETE
Total URLs collected: 210,386
